In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer

In [9]:
path = "../data/raw/PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(path)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 706.2 MB


In [10]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [11]:
df_par = pd.read_parquet('../data/processed/transactions.parquet')
df_par.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 9 columns):
 #   Column          Dtype   
---  ------          -----   
 0   step            int32   
 1   type            category
 2   amount          float32 
 3   nameOrig        string  
 4   oldbalanceOrg   float32 
 5   newbalanceOrig  float32 
 6   nameDest        string  
 7   oldbalanceDest  float32 
 8   newbalanceDest  float32 
dtypes: category(1), float32(5), int32(1), string(2)
memory usage: 376.0 MB


In [12]:
df_par.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest
0,1,PAYMENT,9839.639648,C1231006815,170136.0,160296.359375,M1979787155,0.0,0.0
1,1,PAYMENT,1864.280029,C1666544295,21249.0,19384.720703,M2044282225,0.0,0.0
2,1,TRANSFER,181.000000,C1305486145,181.0,0.000000,C553264065,0.0,0.0
3,1,CASH_OUT,181.000000,C840083671,181.0,0.000000,C38997010,21182.0,0.0
4,1,PAYMENT,11668.139648,C2048537720,41554.0,29885.859375,M1230701703,0.0,0.0


In [13]:
df_par['type'].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

# Preprocessing

In [14]:
def perform_feature_engineering(X):
    df_new = X.copy()
    
    # 1. Hour of Day
    df_new['hour_of_day'] = df_new['step'] % 24
    
    # 2. Check Balance Orig
    df_new['errorBalanceOrig'] = df_new['newbalanceOrig'] + df_new['amount'] - df_new['oldbalanceOrg']
    
    # 3. Check Balance Dest & Flagging Merchant
    df_new['is_merchant_dest'] = df_new['nameDest'].str.startswith('M').astype('int8')
    df_new['errorBalanceDest'] = np.where(
        df_new['is_merchant_dest'] == 1, 0,
        df_new['oldbalanceDest'] + df_new['amount'] - df_new['newbalanceDest']
    )

    # 4. Balance Drain Ratio
    df_new['balance_drain_ratio'] = np.where(df_new['oldbalanceOrg'] > 0, df_new['amount'] / df_new['oldbalanceOrg'], -1)

    # 5. Binning Time Segmentation
    bins_time = [-1, 6, 18, 24]
    labels_time = ['Midnight_to_Morning', 'Working_Hours', 'Evening']
    df_new['time_segment'] = pd.cut(df_new['hour_of_day'], bins=bins_time, labels=labels_time)

    # 6. Binning Amount (qcut)
    labels_amount = ['Low_Amount', 'Medium_Amount', 'High_Amount']
    df_new['amount_category'] = pd.qcut(df_new['amount'], q=3, labels=labels_amount)

    # 7. Drop noise & redundan feature
    cols_to_drop = ['nameOrig', 'nameDest', 'newbalanceDest', 'newbalanceOrig', 'step', 'is_merchant_dest']
    df_new = df_new.drop(columns=cols_to_drop)
    return df_new

In [15]:
def select_segmentation_features(X):
    return X.drop(columns=['time_segment', 'amount_category'])

def select_pattern_features(X):
    return X[['type', 'time_segment', 'amount_category']]

In [16]:
nums_cols = ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'errorBalanceOrig', 'errorBalanceDest', 'balance_drain_ratio', 'hour_of_day']
cat_cols = ['type']

preprocessor_seg = ColumnTransformer(
    transformers = [
        ('num', RobustScaler(), nums_cols),
        ('cat', OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore'), cat_cols)
    ],
    remainder='drop'
)

segmentation_pipeline = Pipeline([
    ('core_eng', FunctionTransformer(perform_feature_engineering)),
    ('selector', FunctionTransformer(select_segmentation_features)),
    ('preprocessing', preprocessor_seg)
])

pattern_pipeline = Pipeline([
    ('core_eng', FunctionTransformer(perform_feature_engineering)),
    ('selector', FunctionTransformer(select_pattern_features))
])

In [17]:
# 1. Clustering
data_seg_array = segmentation_pipeline.fit_transform(df)
feature_names = (
    segmentation_pipeline.named_steps['preprocessing'].transformers_[0][2] + 
    list(segmentation_pipeline.named_steps['preprocessing'].transformers_[1][1].get_feature_names_out(cat_cols))
)
df_clustering_final = pd.DataFrame(data_seg_array, columns=feature_names)
df_clustering_final.to_parquet('../data/processed/data_phase2_clustering.parquet', index=False)

# 2. Apriori
df_rules_final = pattern_pipeline.fit_transform(df)
df_rules_final.to_parquet('../data/processed/data_phase3_rules.parquet', index=False)

In [18]:
df_clustering_final.head(20)

,amount,oldbalanceOrg,oldbalanceDest,errorBalanceOrig,errorBalanceDest,balance_drain_ratio,hour_of_day,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,-0.332932,1.452991,-0.140722,-0.278399,0.000000e+00,-0.005895,-2.142857,0.0,0.0,0.0,1.0,0.0
1,-0.373762,0.065610,-0.140722,-0.278399,0.000000e+00,0.003393,-2.142857,0.0,0.0,0.0,1.0,0.0
2,-0.382380,-0.130708,-0.140722,-0.278399,1.810000e+04,0.286754,-2.142857,0.0,0.0,0.0,0.0,1.0
3,-0.382380,-0.130708,-0.118260,-0.278399,2.136300e+06,0.286754,-2.142857,0.0,1.0,0.0,0.0,0.0
4,-0.323571,0.254820,-0.140722,-0.278399,0.000000e+00,0.063360,-2.142857,0.0,0.0,0.0,1.0,0.0
5,-0.343284,0.369491,-0.140722,-0.278399,0.000000e+00,0.021226,-2.142857,0.0,0.0,0.0,1.0,0.0
6,-0.346918,1.574679,-0.140722,-0.278399,0.000000e+00,-0.011807,-2.142857,0.0,0.0,0.0,1.0,0.0
7,-0.343059,1.508447,-0.140722,-0.278399,0.000000e+00,-0.009991,-2.142857,0.0,0.0,0.0,1.0,0.0
8,-0.362704,-0.107506,-0.140722,-0.272912,0.000000e+00,0.444137,-2.142857,0.0,0.0,0.0,1.0,0.0
9,-0.355980,0.256366,-0.096293,-0.278399,6.886980e+05,0.015882,-2.142857,0.0,0.0,1.0,0.0,0.0


In [19]:
df_rules_final.head()

,type,time_segment,amount_category
0,PAYMENT,Midnight_to_Morning,Low_Amount
1,PAYMENT,Midnight_to_Morning,Low_Amount
2,TRANSFER,Midnight_to_Morning,Low_Amount
3,CASH_OUT,Midnight_to_Morning,Low_Amount
4,PAYMENT,Midnight_to_Morning,Low_Amount
